In [1]:
# 加载所需要的包
%matplotlib inline

# scitnific computing and plotting
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

# HDDM related packages
import pymc as pm
import hddm
import kabuki
import arviz as az
print("The current HDDM version is: ", hddm.__version__)
print("The current kabuki version is: ", kabuki.__version__)
print("The current PyMC version is: ", pm.__version__)
print("The current ArviZ version is: ", az.__version__)

import warnings
warnings.filterwarnings('ignore') 

The current HDDM version is:  1.0.1RC
The current kabuki version is:  0.6.5RC4
The current PyMC version is:  2.3.8
The current ArviZ version is:  0.15.1


In [22]:
data_match = hddm.load_csv('../../../3_Data/exp2/CleanData/data_match.csv')
data_rdk = hddm.load_csv('../../../3_Data/exp2/CleanData/data_rdk.csv')
#剔除缺失值
data_rdk = data_rdk.dropna(subset=['association'])

In [ ]:
# 读取数据并进行预处理，response=choice
# data_color = hddm.load_csv('../../3_Data/exp1a/CleanData/data_color.csv')

data_match = data_match.assign(
    correct=data_match['correct'].astype(int),
    response=data_match['response'].map({'f': 1, 'j': 0}),
    choice=data_match['response'].map({'f': 1, 'j': 0}),
    stim = data_match['matchness'].map({'match':1, 'mismatch':-1}),
    difficulty = data_match['difficulty'].map({'easy':0, 'hard':1})

)

data_rdk = data_rdk.assign(
    correct=data_rdk['correct'].astype(int),
    difficulty = data_rdk['difficulty'].map({'easy':0, 'hard':1}),
    
    # 按键编码：d=1, k=0, 左箭头=1, 右箭头=0
    response=data_rdk['response'].map({
        'd': 1, 
        'k': 0, 
        'arrowleft': 1, 
        'arrowright': 0
    }),
    
    # 运动方向：左180→1，右0→-1
    # coherent_direction=data_rdk['coherent_direction'].map({180: 1, 0: -1}),
    
    # 合并运动方向 + 颜色编码 stim：左和红为1，右和蓝为0
   stim=data_rdk.apply(
    lambda row:
        # 运动任务：只按方向判断
        1 if (row['part'] == 'RDK_motion' and row['coherent_direction'] == 180) else
        0 if (row['part'] == 'RDK_motion' and row['coherent_direction'] == 0) else
        # 颜色任务：只按颜色判断
        1 if (row['part'] == 'RDK_color' and row['dot_color_final'] == '["hsl(0, 50%, 50%)","hsl(225, 50%, 50%)"]') else
        0, # if (row['part'] == 'RDK_colorr' and row['dot_color_final'] == '["hsl(225, 50%, 50%)","hsl(0, 50%, 50%)"]') else # 这里有点问题，不这样写会得空值
        # np.nan,  # 都不满足就标记缺失
    axis=1
   )
)

,subj_idx,rt,correct,coherence,part,response,dot_color_final,coherent_direction,isMatch,association,difficulty,group,task_type,matchness,choice,stim
0,36,1.746,1,0.00,match_RDK,0,"[""hsl(0, 50%, 50%)"",""hsl(225, 50%, 50%)""]",0,mismatch,self,0,color,NaN,mismatch,0,-1
1,36,1.615,1,0.00,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",0,match,other,0,color,NaN,match,1,1
2,36,2.484,1,0.00,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",0,match,other,1,color,NaN,match,1,1
3,36,1.952,0,0.00,match_RDK,0,"[""hsl(0, 50%, 50%)"",""hsl(225, 50%, 50%)""]",0,match,self,1,color,NaN,match,0,1
4,36,2.738,1,0.00,match_RDK,1,"[""hsl(0, 50%, 50%)"",""hsl(225, 50%, 50%)""]",0,match,self,0,color,NaN,match,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9720,9,1.977,1,0.10,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",180,match,other,0,motion,NaN,match,1,1
9721,9,3.718,1,0.10,match_RDK,0,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",180,mismatch,other,0,motion,NaN,mismatch,0,-1
9722,9,2.382,1,0.07,match_RDK,0,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",0,mismatch,self,1,motion,NaN,mismatch,0,-1
9723,9,2.153,1,0.10,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",0,match,self,0,motion,NaN,match,1,1


## 匹配任务

- m0: 基础模型
- m1：v随matchness，association和difficulty变化，dc随associaition变化，z随association变化
- m2：v随matchness，association, difficulty和group变化，dc随associaition变化，z随association变化

In [ ]:
data_match

### m0

In [ ]:
data_match['response'] = data_match['choice']

# 构建回归模型
model_reg = [
    # {'model': "z ~ 1 + isMatch*association", 'link_func': lambda x: x},
    {'model': "v ~ 1", 'link_func': lambda x: x}
] 
 
match_task_m0 = hddm.HDDMRegressor(
    data_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    group_only_regressors=False
)

match_task_m0_idata = match_task_m0.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'match_task_m0.db', db = 'pickle')
match_task_m0.save('match_task_m0.hddm')

### m1

In [ ]:
data_match['response'] = data_match['choice']

# 构建回归模型
model_reg = [
    # {'model': "z ~ 1 + isMatch*association", 'link_func': lambda x: x},
    {'model': "v ~ 1 + association*stim*difficulty", 'link_func': lambda x: x}
] 
 
match_task_m1 = hddm.HDDMRegressor(
    data_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_m1_idata = match_task_m1.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'match_task_m1.db', db = 'pickle')
match_task_m1.save('match_task_m1.hddm')

In [27]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(match_task_m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                            mean     sd  hdi_3%  hdi_97%   
a                                          2.567  0.058   2.462    2.680  \
a_std                                      0.413  0.050   0.322    0.505   
t                                          0.887  0.036   0.819    0.952   
t_std                                      0.274  0.029   0.222    0.328   
z(other)                                   0.540  0.007   0.528    0.554   
z(self)                                    0.555  0.006   0.543    0.567   
z_std                                      0.088  0.019   0.052    0.125   
v_Intercept                               -0.045  0.025  -0.097   -0.003   
v_Intercept_std                            0.072  0.020   0.034    0.108   
v_association[T.self]                     -0.021  0.034  -0.086    0.044   
v_association[T.self]_std                  0.057  0.031   0.001    0.105   
v_stim                                     0.781  0.044   0.703    0.869   
v_stim_std  

### m2

In [ ]:
data_match['response'] = data_match['choice']

# 构建回归模型
model_reg = [
    # {'model': "z ~ 1 + isMatch*association", 'link_func': lambda x: x},
    {'model': "v ~ 1 + association*stim*difficulty*group", 'link_func': lambda x: x}
] 
 
match_task_m2 = hddm.HDDMRegressor(
    data_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_m2_idata = match_task_m2.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'match_task_m2.db', db = 'pickle')
match_task_m2.save('match_task_m2.hddm')

In [ ]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(match_task_m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### 模型比较

In [ ]:
# 加载模型
# match_task_m1 = hddm.load('match_task_m1.hddm')
# match_task_m2 = hddm.load('match_task_m2.hddm')

In [ ]:
DIC_dict = {
    "m0": match_task_m0.dic,
    "m1": match_task_m1.dic,
    "m2": match_task_m2.dic
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)

,model,DIC
m1,m1,29683.956460
m2,m2,29669.163566


## 辨别任务

- m0：基础模型
- m1：v随关联类型和任务类型变化
- m2：v随关联类型、任务类型和难度变化
- m3：v随关联类型、任务类型、难度和组别变化

### m0

In [ ]:
data_rdk['response'] = data_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1", 'link_func': lambda x: x}
] 
 
rdk_task_m0 = hddm.HDDMRegressor(
    data_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_m0_idata = rdk_task_m0.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'rdk_task_m0.db', db = 'pickle')
rdk_task_m0.save('rdk_task_m0.hddm')

### m1

In [ ]:
data_rdk['response'] = data_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*task_type", 'link_func': lambda x: x}
] 
 
rdk_task_m1 = hddm.HDDMRegressor(
    data_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_m1_idata = rdk_task_m1.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'rdk_task_m1.db', db = 'pickle')
rdk_task_m1.save('rdk_task_m1.hddm')

In [ ]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### m2

In [ ]:
data_rdk['response'] = data_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*task_type*difficulty", 'link_func': lambda x: x}
] 
 
rdk_task_m2 = hddm.HDDMRegressor(
    data_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_m2_idata = rdk_task_m2.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'rdk_task_m2.db', db = 'pickle')
rdk_task_m2.save('rdk_task_m2.hddm')

In [48]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   
a                          2.666  0.073   2.540    2.812      0.002    0.001  \
a_std                      0.521  0.063   0.406    0.644      0.002    0.001   
t                          0.834  0.035   0.769    0.901      0.001    0.001   
t_std                      0.257  0.028   0.206    0.311      0.001    0.000   
z                          0.526  0.006   0.513    0.536      0.001    0.001   
z_std                      0.039  0.025   0.000    0.081      0.005    0.004   
v_Intercept                0.538  0.039   0.465    0.609      0.001    0.001   
v_Intercept_std            0.247  0.032   0.190    0.306      0.001    0.001   
v_association[T.self]      0.065  0.060  -0.048    0.174      0.001    0.001   
v_association[T.self]_std  0.411  0.050   0.320    0.504      0.001    0.001   

                           ess_bulk  ess_tail  r_hat  
a                            2033.0    3420.0   1.01  
a_std    

### m3

In [ ]:
data_rdk['response'] = data_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*task_type*difficulty*group", 'link_func': lambda x: x}
] 
 
rdk_task_m3 = hddm.HDDMRegressor(
    data_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_m3_idata = rdk_task_m3.sample(10000, burn=4000, chains=4, return_infdata=True, dbname = 'rdk_task_m3.db', db = 'pickle')
rdk_task_m3.save('rdk_task_m3.hddm')

In [ ]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_m3_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### 模型比较

In [ ]:
# 加载模型
# rdk_task_m1 = hddm.load('rdk_task_m1.hddm')
# rdk_task_m2 = hddm.load('rdk_task_m2.hddm')
rdk_task_m3 = hddm.load('rdk_task_m3.hddm')

DIC_dict = {
    "m0": rdk_task_m0.dic,
    "m1": rdk_task_m1.dic,
    "m2": rdk_task_m2.dic,
    "m3": rdk_task_m3.dic
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)

,model,DIC
m1,m1,29436.550101
m2,m2,29143.194852
m3,m3,29086.842379


### 探索：难度的影响

In [ ]:
# 筛选出任务不相关条件的数据
# data_irrelevant = data_rdk[data_rdk['task_type'] == 'irrelevant']

In [16]:
# # 筛选出任务不相关条件的数据
# data_irrelevant = data_rdk[data_rdk['task_type'] == 'irrelevant']
# # 用回归模型看看交互效应
# v_reg = [{'model': "v ~ association*difficulty", 'link_func': lambda x: x}] 

# hddmReg_v_diff = hddm.HDDMRegressor(
#     data_irrelevant,
#     v_reg,
#     bias=True, p_outlier=0.05,
#     include=['v', 'a', 't', 'z'],
#     group_only_regressors=False
# )


# hddmReg_v_diff_idata = hddmReg_v_diff.sample(2000, burn=500, chains=4, return_infdata=True, dbname = 'hddmReg_v_diff.db', db = 'pickle')
# hddmReg_v_diff.save('hddmReg_v_diff.hddm')

In [ ]:
# # 只筛选出不含"_subj."的参数行
# summary = az.summary(hddmReg_v_diff_idata)
# group_params = summary[~summary.index.str.contains('_subj.')]
# print(group_params)

                                               mean     sd  hdi_3%  hdi_97%   
a                                             6.112  0.629   5.009    7.255  \
a_std                                         1.964  0.310   1.389    2.536   
t                                             0.261  0.022   0.221    0.304   
t_std                                         0.110  0.020   0.074    0.148   
z                                             0.329  0.033   0.271    0.386   
z_std                                         0.051  0.033   0.002    0.112   
v_Intercept                                  -1.668  0.074  -1.808   -1.530   
v_Intercept_std                               0.477  0.059   0.369    0.586   
v_association[T.self]                        -0.049  0.043  -0.125    0.036   
v_association[T.self]_std                     0.043  0.037   0.001    0.112   
v_difficulty[T.hard]                          0.123  0.034   0.059    0.184   
v_difficulty[T.hard]_std                      0.062 